In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

c:\Users\ahren\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import re
import unicodedata
import pandas as pd

def clean_text_light(text):
    if pd.isna(text):
        return ""
    
    text = str(text)

    # normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # replace non-breaking spaces etc.
    text = text.replace("\xa0", " ").replace("\n", " ").replace("\r", " ").replace("\t", " ")

    # remove urls
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # remove common decorative / box-drawing / symbol-like unicode chars
    text = re.sub(r"[▀▌▐■□▪▫●○◆◇★☆♠♣♥♦║═╣╚╔╗╦╩╠╝]+", " ", text)

    # remove emoji / pictograph ranges
    text = re.sub(
        r"[\U0001F300-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF"
        r"\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF"
        r"\U00002700-\U000027BF\U000024C2-\U0001F251]+",
        " ",
        text
    )

    # keep letters, numbers, basic punctuation
    text = re.sub(r"[^A-Za-z0-9\s\.,!?':;\"()\-/&]", " ", text)

    # collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

train_data = pd.read_csv("training_data.csv")
train_label = pd.read_csv("training_label.csv")

valid_data = pd.read_csv("validation_data.csv")
valid_label = pd.read_csv("validation_label.csv")

test_data = pd.read_csv("testing_data.csv")
test_label = pd.read_csv("testing_label.csv")

for df in [train_data, valid_data, test_data]:
    df["video_title"] = df["video_title"].fillna("").astype(str)
    df["video_description"] = df["video_description"].fillna("").astype(str)

    # light-cleaned version
    df["video_title_clean"] = df["video_title"].apply(clean_text_light)
    df["video_description_clean"] = df["video_description"].apply(clean_text_light)


train_df = pd.concat([train_data, train_label], axis=1)
valid_df = pd.concat([valid_data, valid_label], axis=1)
test_df  = pd.concat([test_data,  test_label], axis=1)

for df in [train_df, valid_df, test_df]:
    df["category"] = df["category"].astype(str).str.strip()

print(train_df.head())
print(train_df["category"].value_counts())

                                         video_title  \
0                           Most annoying skill ever   
1                         Levy STUNS World Champion!   
2                    Nothing gets past Terry Crews 👀   
3  Private Room on Canada’s Overnight Sleeper Tra...   
4                                                Jew   

                                   video_description  \
0                                                      
1  ➡️ Get My Chess Courses:  https://www.chessly....   
2                                                      
3  To get a 1 year supply of immune-supporting Vi...   
4                                                      

                                   video_title_clean  \
0                           Most annoying skill ever   
1                         Levy STUNS World Champion!   
2                      Nothing gets past Terry Crews   
3  Private Room on Canada s Overnight Sleeper Tra...   
4                                             

In [4]:
labels = sorted(train_df["category"].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

for df in [train_df, valid_df, test_df]:
    df["labels"] = df["category"].map(label2id)

print(label2id)
print("Number of classes:", len(labels))

{'Business': 0, 'Digital Media': 1, 'Education': 2, 'Food & Cooking': 3, 'Health': 4, 'Lifestyle': 5, 'Music': 6, 'News': 7, 'Shopping': 8, 'Sports & Games': 9, 'Vehicle': 10, 'Video Games': 11, 'nan': 12}
Number of classes: 13


In [5]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[["video_title_clean", "video_description_clean", "labels"]],
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        valid_df[["video_title_clean", "video_description_clean", "labels"]],
        preserve_index=False
    ),
    "test": Dataset.from_pandas(
        test_df[["video_title_clean", "video_description_clean", "labels"]],
        preserve_index=False
    )
})

dataset

DatasetDict({
    train: Dataset({
        features: ['video_title_clean', 'video_description_clean', 'labels'],
        num_rows: 1480
    })
    validation: Dataset({
        features: ['video_title_clean', 'video_description_clean', 'labels'],
        num_rows: 92
    })
    test: Dataset({
        features: ['video_title_clean', 'video_description_clean', 'labels'],
        num_rows: 279
    })
})

In [6]:
MODEL_2_NAME = "microsoft/MiniLM-L12-H384-uncased"

tokenizer_2 = AutoTokenizer.from_pretrained(MODEL_2_NAME)

model_2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_2_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

c:\Users\ahren\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ahren\.cache\huggingface\hub\models--microsoft--MiniLM-L12-H384-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 29005.65it/s]
BertForSeq

In [7]:
def processing_class_2(examples):
    return tokenizer_2(
        examples["video_title_clean"],
        examples["video_description_clean"],
        truncation=True,
        max_length=256
    )

tokenized_dataset_2 = dataset.map(processing_class_2, batched=True)

columns_to_keep_2 = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in tokenized_dataset_2["train"].column_names:
    columns_to_keep_2.append("token_type_ids")

for split in ["train", "validation", "test"]:
    tokenized_dataset_2[split] = tokenized_dataset_2[split].remove_columns(
        [col for col in tokenized_dataset_2[split].column_names if col not in columns_to_keep_2]
    )

data_collator_2 = DataCollatorWithPadding(tokenizer=tokenizer_2)

tokenized_dataset_2

Map: 100%|██████████| 279/279 [00:00<00:00, 10260.60 examples/s]


DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1480
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 92
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 279
    })
})

In [8]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(labels)),
    y=train_df["labels"].values
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

tensor([ 7.1154,  1.1385,  0.3069,  0.9409, 12.6496,  0.3603,  2.9191,  1.6499,
         1.0740,  0.7851,  2.0330,  0.8757, 56.9231])


In [9]:
def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels_true, preds, average="macro", zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels_true, preds, average="weighted", zero_division=0
    )

    acc = accuracy_score(labels_true, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }

In [10]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [15]:
training_args_2 = TrainingArguments(
    output_dir="./results_model_2_minilm",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    dataloader_num_workers=0,
    fp16=False,
    seed=42
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [16]:
trainer_2 = WeightedTrainer(
    model=model_2,
    args=training_args_2,
    train_dataset=tokenized_dataset_2["train"],
    eval_dataset=tokenized_dataset_2["validation"],
    processing_class=tokenizer_2,
    data_collator=data_collator_2,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer_2.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,3.971200,2.044221,0.358696,0.344759,0.462482,0.379433,0.293882,0.358696,0.310900
2,3.567515,1.915127,0.380435,0.397817,0.506926,0.416253,0.338765,0.380435,0.337912
3,3.129635,1.925619,0.347826,0.368982,0.454690,0.376078,0.317235,0.347826,0.312489
4,2.826711,1.914367,0.456522,0.487105,0.471884,0.443453,0.512414,0.456522,0.432171
5,2.502174,1.807922,0.467391,0.517566,0.509879,0.483505,0.514624,0.467391,0.429290
6,2.234005,1.820440,0.500000,0.497505,0.513764,0.483314,0.548005,0.500000,0.478952
7,2.056547,1.822768,0.478261,0.470487,0.497025,0.470426,0.503177,0.478261,0.464484


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.09it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=651, training_loss=2.898255332091254, metrics={'train_runtime': 1672.4266, 'train_samples_per_second': 8.849, 'train_steps_per_second': 0.556, 'total_flos': 185876026206528.0, 'train_loss': 2.898255332091254, 'epoch': 7.0})

In [18]:
validation_results_2 = trainer_2.predict(tokenized_dataset_2["validation"])
test_results_2 = trainer_2.predict(tokenized_dataset_2["test"])

print("Validation results:", validation_results_2.metrics)
print("Test results:", test_results_2.metrics)

Validation results: {'test_loss': 1.8073608875274658, 'test_accuracy': 0.4673913043478261, 'test_precision_macro': 0.5124098124098124, 'test_recall_macro': 0.5098790098790098, 'test_f1_macro': 0.48049146931499875, 'test_precision_weighted': 0.5121808143547274, 'test_recall_weighted': 0.4673913043478261, 'test_f1_weighted': 0.42802368773596394, 'test_runtime': 3.2865, 'test_samples_per_second': 27.993, 'test_steps_per_second': 3.651}
Test results: {'test_loss': 1.8104263544082642, 'test_accuracy': 0.5519713261648745, 'test_precision_macro': 0.5213362176918453, 'test_recall_macro': 0.4989801420822588, 'test_f1_macro': 0.4827998690635829, 'test_precision_weighted': 0.5666556841342694, 'test_recall_weighted': 0.5519713261648745, 'test_f1_weighted': 0.5261459282979575, 'test_runtime': 11.8853, 'test_samples_per_second': 23.474, 'test_steps_per_second': 2.945}


In [20]:
test_predictions = trainer_2.predict(tokenized_dataset_2["test"])
y_pred = np.argmax(test_predictions.predictions, axis=1)
y_true = test_df["labels"].values

print("Prediction logits shape:", test_predictions.predictions.shape)
print("Unique y_true:", sorted(set(y_true)))
print("Unique y_pred:", sorted(set(y_pred)))
print("Label names:", len(labels))

all_label_ids = list(range(len(labels)))

print(classification_report(
    y_true,
    y_pred,
    labels=all_label_ids,
    target_names=labels,
    digits=4,
    zero_division=0
))

Prediction logits shape: (279, 13)
Unique y_true: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]
Unique y_pred: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]
Label names: 13
                precision    recall  f1-score   support

      Business     0.6667    0.2500    0.3636         8
 Digital Media     0.2391    0.4783    0.3188        23
     Education     0.6087    0.7368    0.6667        76
Food & Cooking     0.7812    0.8333    0.8065        30
        Health     0.0000    0.0000    0.0000         3
     Lifestyle     0.5556    0.1111    0.1852        45
         Music     0.4000    0.2857    0.3333         7
          News     0.4444    0.4000    0.4211        20
      Shopping     0.6429    0.5625    0.6000        16
Sports & Games     0.6190    0.6500    0.6341        

In [21]:
trainer_2.save_model("./final_minilm_model")
tokenizer_2.save_pretrained("./final_minilm_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.51it/s]


('./final_minilm_model\\tokenizer_config.json',
 './final_minilm_model\\tokenizer.json')